# Machine Learning & Predictive Modeling
## Insurance Risk Analytics - Task 4

This notebook builds and evaluates predictive models for dynamic, risk-based pricing.

## Objectives

1. **Claim Severity Prediction**: Build models to predict TotalClaims (using charges as proxy)
2. **Premium Optimization**: Develop ML models to predict optimal premium values
3. **Regional Models**: Build linear regression models per region (adapting from zipcode requirement)
4. **Model Evaluation**: Compare multiple algorithms (Linear Regression, Decision Trees, Random Forest, XGBoost)
5. **Feature Importance**: Analyze which features drive predictions using SHAP values
6. **Business Insights**: Provide actionable recommendations for risk-based pricing

## Data Adaptations

**Note**: The current dataset structure:
- **Available**: age, sex, bmi, children, smoker, region, charges
- **Missing**: TotalClaims, TotalPremium, CalculatedPremiumPerTerm, zip codes

**Adaptations**:
- Use **charges** as proxy for TotalClaims (claim severity)
- Use **charges** as proxy for premium (risk-based pricing)
- Create synthetic **claim indicator** (binary: has_claim) based on charges threshold
- Build regional models instead of zipcode models (4 regions available)


## 1. Setup and Data Loading


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import sys
import os

# Machine Learning imports
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# SHAP for interpretability
import shap

# Suppress warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")


In [ ]:
# Add src to path - robust path resolution
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

src_path = project_root / 'src'
if src_path.exists():
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))
    print(f"✓ Added to path: {src_path}")
else:
    print(f"⚠ Warning: Could not find src directory")

# Import custom modules
try:
    from data_loader import load_insurance_data
    from ml_utils import (
        prepare_features,
        train_linear_regression,
        train_decision_tree,
        train_random_forest,
        train_xgboost,
        get_feature_importance,
        calculate_shap_values,
        compare_models,
        create_claim_indicator
    )
    print("✓ Custom modules imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

# Create figures directory
figures_dir = project_root / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Figures directory: {figures_dir}")


In [ ]:
# Load data
df = load_insurance_data()
print(f"✓ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Info:")
print(df.info())
print(f"\nFirst few rows:")
df.head()


## 2. Data Preparation and Feature Engineering


In [ ]:
# Create claim indicator (binary: has_claim)
# Since we don't have explicit claim data, we'll create a synthetic indicator
# where claims > 0 if charges exceed median (simulating claim occurrence)
df = create_claim_indicator(df, charges_col='charges')

print("✓ Claim indicator created")
print(f"\nClaim Distribution:")
print(df['has_claim'].value_counts())
print(f"\nClaim Rate: {df['has_claim'].mean():.2%}")

# Feature engineering
# Create additional features that might be relevant
df['age_squared'] = df['age'] ** 2
df['bmi_category'] = pd.cut(df['bmi'], 
                             bins=[0, 18.5, 25, 30, 100],
                             labels=['underweight', 'normal', 'overweight', 'obese'])
df['age_group'] = pd.cut(df['age'],
                         bins=[0, 30, 40, 50, 100],
                         labels=['young', 'middle', 'senior', 'elderly'])

print("\n✓ Feature engineering completed")
print(f"\nNew columns: {[col for col in df.columns if col not in ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']]}")


In [ ]:
# Define features for modeling
categorical_cols = ['sex', 'smoker', 'region', 'bmi_category', 'age_group']
numerical_cols = ['age', 'bmi', 'children', 'age_squared']

# Check for missing values
print("Missing values check:")
print(df[categorical_cols + numerical_cols + ['charges', 'has_claim']].isnull().sum())

# Display data summary
print("\n" + "="*70)
print("DATA SUMMARY FOR MODELING")
print("="*70)
print(f"Total records: {len(df)}")
print(f"Features: {len(categorical_cols + numerical_cols)}")
print(f"  - Categorical: {len(categorical_cols)}")
print(f"  - Numerical: {len(numerical_cols)}")
print(f"Target variable: charges (for regression)")
print(f"Claim indicator: has_claim (for classification)")
print("="*70)


## 3. Claim Severity Prediction (Risk Model)

**Objective**: Predict TotalClaims (using charges as proxy) for policies that have claims.

**Target Variable**: `charges` (for subset where `has_claim == 1`)  
**Evaluation Metrics**: RMSE, R², MAE


In [ ]:
# Prepare data for claim severity prediction
# Use only records with claims (has_claim == 1)
df_claims = df[df['has_claim'] == 1].copy()

print(f"Records with claims: {len(df_claims)} ({len(df_claims)/len(df):.1%} of total)")
print(f"\nCharges statistics for claims subset:")
print(df_claims['charges'].describe())

# Prepare features
X_train_claims, X_test_claims, y_train_claims, y_test_claims, feature_names_claims, encoders_claims = prepare_features(
    df_claims,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    target_col='charges',
    test_size=0.2,
    random_state=42,
    encode_method='onehot'
)

print(f"\n✓ Data prepared for claim severity prediction")
print(f"Training set: {len(X_train_claims)} samples")
print(f"Test set: {len(X_test_claims)} samples")
print(f"Features: {len(feature_names_claims)}")


### 3.1 Train Multiple Models for Claim Severity


In [ ]:
# Store models and results
claim_models = {}
claim_results = {}

# 1. Linear Regression
print("Training Linear Regression...")
model_lr, metrics_lr = train_linear_regression(
    X_train_claims, y_train_claims, X_test_claims, y_test_claims
)
claim_models['Linear Regression'] = model_lr
claim_results['Linear Regression'] = metrics_lr
print(f"  Test RMSE: ${metrics_lr['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_lr['test_r2']:.4f}")

# 2. Decision Tree
print("\nTraining Decision Tree...")
model_dt, metrics_dt = train_decision_tree(
    X_train_claims, y_train_claims, X_test_claims, y_test_claims,
    max_depth=10, min_samples_split=5
)
claim_models['Decision Tree'] = model_dt
claim_results['Decision Tree'] = metrics_dt
print(f"  Test RMSE: ${metrics_dt['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_dt['test_r2']:.4f}")

# 3. Random Forest
print("\nTraining Random Forest...")
model_rf, metrics_rf = train_random_forest(
    X_train_claims, y_train_claims, X_test_claims, y_test_claims,
    n_estimators=100, max_depth=10
)
claim_models['Random Forest'] = model_rf
claim_results['Random Forest'] = metrics_rf
print(f"  Test RMSE: ${metrics_rf['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_rf['test_r2']:.4f}")

# 4. XGBoost
print("\nTraining XGBoost...")
model_xgb, metrics_xgb = train_xgboost(
    X_train_claims, y_train_claims, X_test_claims, y_test_claims,
    n_estimators=100, max_depth=6, learning_rate=0.1
)
claim_models['XGBoost'] = model_xgb
claim_results['XGBoost'] = metrics_xgb
print(f"  Test RMSE: ${metrics_xgb['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_xgb['test_r2']:.4f}")

print("\n✓ All models trained for claim severity prediction")


In [ ]:
# Compare models
comparison_claims = compare_models(claim_results)
print("="*70)
print("CLAIM SEVERITY MODEL COMPARISON")
print("="*70)
print(comparison_claims.to_string(index=False))
print("="*70)

# Identify best model
best_model_name_claims = comparison_claims.iloc[0]['Model']
best_model_claims = claim_models[best_model_name_claims]
print(f"\n✓ Best Model: {best_model_name_claims}")
print(f"  Test RMSE: ${comparison_claims.iloc[0]['Test RMSE']:,.2f}")
print(f"  Test R²: {comparison_claims.iloc[0]['Test R²']:.4f}")


## 4. Premium Optimization (Pricing Framework)

**Objective**: Develop ML models to predict optimal premium values (using charges as proxy).

**Target Variable**: `charges` (all records)  
**Evaluation Metrics**: RMSE, R², MAE


In [ ]:
# Prepare data for premium prediction (use all records)
X_train_premium, X_test_premium, y_train_premium, y_test_premium, feature_names_premium, encoders_premium = prepare_features(
    df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    target_col='charges',
    test_size=0.2,
    random_state=42,
    encode_method='onehot'
)

print(f"✓ Data prepared for premium prediction")
print(f"Training set: {len(X_train_premium)} samples")
print(f"Test set: {len(X_test_premium)} samples")
print(f"Features: {len(feature_names_premium)}")


In [ ]:
# Store models and results
premium_models = {}
premium_results = {}

# 1. Linear Regression
print("Training Linear Regression...")
model_lr_prem, metrics_lr_prem = train_linear_regression(
    X_train_premium, y_train_premium, X_test_premium, y_test_premium
)
premium_models['Linear Regression'] = model_lr_prem
premium_results['Linear Regression'] = metrics_lr_prem
print(f"  Test RMSE: ${metrics_lr_prem['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_lr_prem['test_r2']:.4f}")

# 2. Decision Tree
print("\nTraining Decision Tree...")
model_dt_prem, metrics_dt_prem = train_decision_tree(
    X_train_premium, y_train_premium, X_test_premium, y_test_premium,
    max_depth=10, min_samples_split=5
)
premium_models['Decision Tree'] = model_dt_prem
premium_results['Decision Tree'] = metrics_dt_prem
print(f"  Test RMSE: ${metrics_dt_prem['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_dt_prem['test_r2']:.4f}")

# 3. Random Forest
print("\nTraining Random Forest...")
model_rf_prem, metrics_rf_prem = train_random_forest(
    X_train_premium, y_train_premium, X_test_premium, y_test_premium,
    n_estimators=100, max_depth=10
)
premium_models['Random Forest'] = model_rf_prem
premium_results['Random Forest'] = metrics_rf_prem
print(f"  Test RMSE: ${metrics_rf_prem['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_rf_prem['test_r2']:.4f}")

# 4. XGBoost
print("\nTraining XGBoost...")
model_xgb_prem, metrics_xgb_prem = train_xgboost(
    X_train_premium, y_train_premium, X_test_premium, y_test_premium,
    n_estimators=100, max_depth=6, learning_rate=0.1
)
premium_models['XGBoost'] = model_xgb_prem
premium_results['XGBoost'] = metrics_xgb_prem
print(f"  Test RMSE: ${metrics_xgb_prem['test_rmse']:,.2f}")
print(f"  Test R²: {metrics_xgb_prem['test_r2']:.4f}")

print("\n✓ All models trained for premium prediction")


In [ ]:
# Compare models
comparison_premium = compare_models(premium_results)
print("="*70)
print("PREMIUM PREDICTION MODEL COMPARISON")
print("="*70)
print(comparison_premium.to_string(index=False))
print("="*70)

# Identify best model
best_model_name_premium = comparison_premium.iloc[0]['Model']
best_model_premium = premium_models[best_model_name_premium]
print(f"\n✓ Best Model: {best_model_name_premium}")
print(f"  Test RMSE: ${comparison_premium.iloc[0]['Test RMSE']:,.2f}")
print(f"  Test R²: {comparison_premium.iloc[0]['Test R²']:.4f}")


## 5. Regional Linear Regression Models

**Objective**: Build linear regression models per region (adapting from zipcode requirement).

Since we don't have zip code data, we'll build separate models for each of the 4 regions.


In [ ]:
# Build linear regression models for each region
regional_models = {}
regional_results = {}

regions = df['region'].unique()
print(f"Building models for {len(regions)} regions: {regions}")

for region in regions:
    print(f"\n{'='*70}")
    print(f"Region: {region.upper()}")
    print(f"{'='*70}")
    
    # Filter data for this region
    df_region = df[df['region'] == region].copy()
    print(f"Records: {len(df_region)}")
    
    # Prepare features (simpler feature set for regional models)
    cat_cols_regional = ['sex', 'smoker']
    num_cols_regional = ['age', 'bmi', 'children']
    
    X_train_reg, X_test_reg, y_train_reg, y_test_reg, feat_names_reg, encoders_reg = prepare_features(
        df_region,
        categorical_cols=cat_cols_regional,
        numerical_cols=num_cols_regional,
        target_col='charges',
        test_size=0.2,
        random_state=42,
        encode_method='onehot'
    )
    
    # Train linear regression
    model_reg, metrics_reg = train_linear_regression(
        X_train_reg, y_train_reg, X_test_reg, y_test_reg
    )
    
    regional_models[region] = model_reg
    regional_results[region] = {
        'n_samples': len(df_region),
        'n_train': len(X_train_reg),
        'n_test': len(X_test_reg),
        **metrics_reg
    }
    
    print(f"  Test RMSE: ${metrics_reg['test_rmse']:,.2f}")
    print(f"  Test R²: {metrics_reg['test_r2']:.4f}")
    print(f"  Test MAE: ${metrics_reg['test_mae']:,.2f}")

print(f"\n✓ Regional models completed for all {len(regions)} regions")


In [ ]:
# Create summary table for regional models
regional_summary = []
for region, results in regional_results.items():
    regional_summary.append({
        'Region': region.capitalize(),
        'Samples': results['n_samples'],
        'Test RMSE': results['test_rmse'],
        'Test R²': results['test_r2'],
        'Test MAE': results['test_mae']
    })

regional_df = pd.DataFrame(regional_summary)
print("="*70)
print("REGIONAL LINEAR REGRESSION MODELS SUMMARY")
print("="*70)
print(regional_df.to_string(index=False))
print("="*70)


## 6. Feature Importance Analysis

Analyze which features are most influential in predicting charges (premium/claims).


In [ ]:
# Get feature importance from best premium model
feature_importance = get_feature_importance(
    best_model_premium,
    feature_names_premium,
    top_n=15
)

print("="*70)
print(f"TOP 15 FEATURE IMPORTANCE ({best_model_name_premium})")
print("="*70)
print(feature_importance.to_string(index=False))
print("="*70)

# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.barh(range(len(feature_importance)), feature_importance['importance'].values)
plt.yticks(range(len(feature_importance)), feature_importance['feature'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.title(f'Top 15 Feature Importance - {best_model_name_premium}', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()

fig_path = figures_dir / 'feature_importance.png'
plt.savefig(str(fig_path), dpi=300, bbox_inches='tight')
print(f"✓ Saved figure: {fig_path}")
plt.show()


## 7. SHAP Analysis for Model Interpretability

Use SHAP (SHapley Additive exPlanations) to understand how individual features influence predictions.


In [ ]:
# Calculate SHAP values for best premium model
# Use a sample of test data for SHAP calculation (for performance)
X_sample = X_test_premium.sample(n=min(100, len(X_test_premium)), random_state=42)

print(f"Calculating SHAP values for {len(X_sample)} samples...")
print(f"Model: {best_model_name_premium}")

try:
    # Calculate SHAP values
    shap_values, explainer = calculate_shap_values(best_model_premium, X_sample, max_samples=100)
    
    # Get feature names
    if isinstance(X_sample, pd.DataFrame):
        feature_names_shap = X_sample.columns.tolist()
    else:
        feature_names_shap = feature_names_premium
    
    print("✓ SHAP values calculated successfully")
    
    # Calculate mean absolute SHAP values for feature importance
    if isinstance(shap_values, list):
        shap_values_array = np.array(shap_values)
    else:
        shap_values_array = shap_values
    
    mean_abs_shap = np.abs(shap_values_array).mean(axis=0)
    shap_importance = pd.DataFrame({
        'feature': feature_names_shap,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False).head(10)
    
    print("\n" + "="*70)
    print("TOP 10 FEATURES BY MEAN ABSOLUTE SHAP VALUE")
    print("="*70)
    print(shap_importance.to_string(index=False))
    print("="*70)
    
except Exception as e:
    print(f"⚠ Error calculating SHAP values: {e}")
    print("This may occur with certain model types. Continuing with feature importance analysis.")
    shap_values = None
    explainer = None


In [ ]:
# Visualize SHAP summary plot if available
if shap_values is not None and explainer is not None:
    try:
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
        plt.tight_layout()
        
        fig_path = figures_dir / 'shap_summary.png'
        plt.savefig(str(fig_path), dpi=300, bbox_inches='tight')
        print(f"✓ Saved SHAP summary plot: {fig_path}")
        plt.show()
        
        # SHAP waterfall plot for a single prediction (example)
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(explainer(X_sample.iloc[[0]]), show=False)
        plt.tight_layout()
        
        fig_path = figures_dir / 'shap_waterfall.png'
        plt.savefig(str(fig_path), dpi=300, bbox_inches='tight')
        print(f"✓ Saved SHAP waterfall plot: {fig_path}")
        plt.show()
        
    except Exception as e:
        print(f"⚠ Error creating SHAP plots: {e}")
else:
    print("⚠ SHAP visualization skipped (values not available)")


## 8. Business Recommendations and Insights

Based on the model analysis, provide actionable recommendations for risk-based pricing.


In [ ]:
print("="*80)
print("BUSINESS RECOMMENDATIONS - MACHINE LEARNING INSIGHTS")
print("="*80)

# 1. Model Performance Summary
print("\n1. MODEL PERFORMANCE SUMMARY")
print("-" * 80)
print(f"Best Premium Prediction Model: {best_model_name_premium}")
print(f"  - Test RMSE: ${comparison_premium.iloc[0]['Test RMSE']:,.2f}")
print(f"  - Test R²: {comparison_premium.iloc[0]['Test R²']:.4f}")
print(f"  - Interpretation: Model explains {comparison_premium.iloc[0]['Test R²']:.1%} of variance in charges")

# 2. Top Features Impact
print("\n2. TOP FEATURES DRIVING PREMIUM PREDICTIONS")
print("-" * 80)
top_features = feature_importance.head(10)
for idx, row in top_features.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

# 3. Regional Model Insights
print("\n3. REGIONAL MODEL PERFORMANCE")
print("-" * 80)
best_region = regional_df.loc[regional_df['Test R²'].idxmax()]
worst_region = regional_df.loc[regional_df['Test R²'].idxmin()]
print(f"  Best performing region: {best_region['Region']} (R² = {best_region['Test R²']:.4f})")
print(f"  Most challenging region: {worst_region['Region']} (R² = {worst_region['Test R²']:.4f})")
print(f"  Recommendation: Consider region-specific pricing models for better accuracy")

# 4. Feature Engineering Insights
print("\n4. FEATURE ENGINEERING INSIGHTS")
print("-" * 80)
if 'smoker' in str(feature_importance['feature'].values):
    print("  ✓ Smoking status is a critical risk factor - significant premium adjustment warranted")
if 'age' in str(feature_importance['feature'].values) or 'age_squared' in str(feature_importance['feature'].values):
    print("  ✓ Age is a key predictor - consider age-based pricing tiers")
if 'bmi' in str(feature_importance['feature'].values):
    print("  ✓ BMI impacts risk - health-based pricing may be beneficial")

# 5. Model Deployment Recommendations
print("\n5. DEPLOYMENT RECOMMENDATIONS")
print("-" * 80)
print(f"  • Use {best_model_name_premium} for production premium prediction")
print("  • Implement regional models for location-specific risk assessment")
print("  • Monitor model performance quarterly and retrain with new data")
print("  • Consider ensemble approach combining multiple models for robustness")
print("  • Implement SHAP-based explanations for customer transparency")

# 6. Risk-Based Pricing Framework
print("\n6. RISK-BASED PRICING FRAMEWORK")
print("-" * 80)
print("  Premium = Base Premium × Risk Multipliers")
print("  Risk Multipliers based on:")
for idx, row in top_features.head(5).iterrows():
    feature_name = row['feature']
    if 'smoker' in feature_name.lower():
        print(f"    • Smoking Status: High impact ({row['importance']:.4f})")
    elif 'age' in feature_name.lower():
        print(f"    • Age: Significant impact ({row['importance']:.4f})")
    elif 'bmi' in feature_name.lower():
        print(f"    • BMI: Moderate impact ({row['importance']:.4f})")
    elif 'region' in feature_name.lower():
        print(f"    • Region: Location-based risk ({row['importance']:.4f})")

print("\n" + "="*80)
print("END OF BUSINESS RECOMMENDATIONS")
print("="*80)
